**Laboratorio #3 - Clasificación de texto**

Juan Diego Letona Aguilar

*20230285*

In [ ]:
#parcialmente reciclado del notebook 1 y 2

import sys
import subprocess

packages = [
    "pandas",
    "numpy",
    "nltk",
    "matplotlib",
    "wordcloud",
    "spacy",
    "tqdm"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", *packages
])

# Modelo pequeño de spaCy para español
subprocess.check_call([
    sys.executable, "-m", "spacy", "download", "es_core_news_sm"
])

0

In [3]:
from pathlib import Path
from collections import Counter
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

#voy a usar spacy porque el corpus es en español y spacy tiene un modelo para español
#otros modelos suelen funcionar mejor para el idioma ingles
#depende del caso de uso

import spacy
from tqdm.auto import tqdm
from wordcloud import WordCloud

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

#recursos de NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

print("Librerías cargadas correctamente.")

c:\Users\juanl\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerías cargadas correctamente.


In [4]:
import pandas as pd

df = pd.read_csv("df_total.csv")
#leemos el archivo csv y lo guardamos en un DataFrame de pandas

print("Dimensiones del DataFrame:", df.shape)
display(df.head())

Dimensiones del DataFrame: (1217, 3)


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresarial para el desarrollo sostenible el director de sostenibilidad y clientes globales de BBVA en Colomb...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domingo que buscará una cooperación más estrecha con su par estadounidense y que apoyará las salidas a bo...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como lo es la aviación Viva presentó su avión rosado A320NEO que apuesta por la equidad de género la luc...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión número 56 de la Convención Bancaria. Este será el primer encuentro de los banqueros del país con el p...,Otra


In [5]:
#configuracion y descripcion de columnas

URL_COL = "url"
TEXT_COL = "news"
CATEGORY_COL = "Type"

descripcion = {
    URL_COL: "enlace original de la noticia",
    TEXT_COL: "contenido textual de la noticia",
    CATEGORY_COL: "categoria asignada a la noticia"
}

columnas = pd.DataFrame({
    "columna": descripcion.keys(),
    "tipo_de_dato": [str(df[col].dtype) for col in descripcion],
    "informacion": descripcion.values()
})

display(
    columnas.style
    .hide(axis="index")
    .set_caption("Columnas del corpus")
)

columna,tipo_de_dato,informacion
url,str,enlace original de la noticia
news,str,contenido textual de la noticia
Type,str,categoria asignada a la noticia


In [6]:
#preparamos el corpus para el procesamiento

corpus = df[[URL_COL, TEXT_COL, CATEGORY_COL]].copy()

corpus[TEXT_COL] = corpus[TEXT_COL].fillna("").astype(str)
corpus = corpus[corpus[TEXT_COL].str.strip().ne("")]
corpus = corpus.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

stopwords_es = set(stopwords.words("spanish"))

#basicamente quitamos la puntuacion de los tokens, para que no afecte el conteo de palabras y tipos de palabras

def quitar_puntuacion(token):
    return "".join(
        caracter for caracter in token
        if not unicodedata.category(caracter).startswith("P")
    ).strip()

def contar_tokens_tipos(columna):
    tokens = [token for documento in columna for token in documento]
    return len(tokens), len(set(tokens))

print("Documentos que serán procesados:", len(corpus))

Documentos que serán procesados: 1137


In [7]:
#tokenizacion

tqdm.pandas(desc="Tokenizando")

corpus["tokens_tokenizados"] = corpus[TEXT_COL].progress_apply(
    lambda texto: word_tokenize(texto, language="spanish")
)

#minusculas

corpus["tokens_minusculas"] = corpus["tokens_tokenizados"].apply(
    lambda tokens: [token.lower() for token in tokens]
)

#eliminacion de puntuacion

corpus["tokens_sin_puntuacion"] = corpus["tokens_minusculas"].apply(
    lambda tokens: [
        limpio for token in tokens
        if (limpio := quitar_puntuacion(token))
    ]
)

#eliminacion de stopwords

corpus["tokens_sin_stopwords"] = corpus["tokens_sin_puntuacion"].apply(
    lambda tokens: [token for token in tokens if token not in stopwords_es]
)

Tokenizando: 100%|██████████| 1137/1137 [00:04<00:00, 235.18it/s]


In [8]:
#lematizacion
nlp = spacy.load(
    "es_core_news_sm",
    disable=["parser", "ner"]
)

textos = corpus["tokens_sin_stopwords"].apply(" ".join)

corpus["tokens_lematizados"] = [
    [
        token.lemma_.lower()
        for token in documento
        if token.lemma_.strip()
    ]
    for documento in tqdm(
        nlp.pipe(textos, batch_size=64),
        total=len(textos),
        desc="Lematizando"
    )
]

Lematizando: 100%|██████████| 1137/1137 [00:36<00:00, 31.50it/s]


In [9]:
#texto normalizado por documento

corpus["texto_normalizado"] = corpus["tokens_lematizados"].apply(
    lambda tokens: " ".join(tokens)
)

display(
    corpus[
        [
            TEXT_COL,
            CATEGORY_COL,
            "tokens_lematizados",
            "texto_normalizado"
        ]
    ].head()
)

,news,Type,tokens_lematizados,texto_normalizado
0,Durante el foro La banca articulador empresarial para el desarrollo sostenible el director de sostenibilidad y clientes globales de BBVA en Colomb...,Otra,"[foro, banca, articulador, empresarial, desarrollo, sostenible, director, sostenibilidad, cliente, global, bbva, colombia, andrés, garcía, asegura...",foro banca articulador empresarial desarrollo sostenible director sostenibilidad cliente global bbva colombia andrés garcía asegurar importante en...
1,El regulador de valores de China dijo el domingo que buscará una cooperación más estrecha con su par estadounidense y que apoyará las salidas a bo...,Regulaciones,"[regulador, valor, chino, decir, domingo, buscar, cooperación, estrecho, par, estadounidense, apoyar, salida, bolso, extranjero, luego, supervisor...",regulador valor chino decir domingo buscar cooperación estrecho par estadounidense apoyar salida bolso extranjero luego supervisor estadounidense ...
2,En una industria históricamente masculina como lo es la aviación Viva presentó su avión rosado A320NEO que apuesta por la equidad de género la luc...,Alianzas,"[industria, históricamente, masculino, aviación, vivo, presentar, avión, rosado, a320neo, apostar, equidad, género, lucha, cáncer, mama, inclusión...",industria históricamente masculino aviación vivo presentar avión rosado a320neo apostar equidad género lucha cáncer mama inclusión diversidaddesde...
3,Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual...,Macroeconomia,"[dato, marzo, ipc, interanual, encadenar, decimoquinto, tasa, positivo, consecutivo, inflación, publicado, ine, mantenido, igual, respecto, avance...",dato marzo ipc interanual encadenar decimoquinto tasa positivo consecutivo inflación publicado ine mantenido igual respecto avance 30 marzo situar...
4,Ayer en Cartagena se dio inicio a la versión número 56 de la Convención Bancaria. Este será el primer encuentro de los banqueros del país con el p...,Otra,"[ayer, cartagena, dar, inicio, versión, número, 56, convención, bancario, primero, encuentro, banquero, país, presidente, gustavo, petro, sesión, ...",ayer cartagena dar inicio versión número 56 convención bancario primero encuentro banquero país presidente gustavo petro sesión clausura próximo v...


**1. Conjuntos de entrenamiento, validación y prueba**

In [10]:
from sklearn.model_selection import train_test_split

#preparamos el texto normalizado y las categorias para el aprendizaje supervisado

X = corpus["texto_normalizado"]
y = corpus[CATEGORY_COL]

print("Numero total de documentos:", len(X))
print("Numero de categorias:", y.nunique())

Numero total de documentos: 1137
Numero de categorias: 7


In [11]:
#dividimos primero el corpus en 70% entrenamiento y 30% temporal

X_entrenamiento, X_temporal, y_entrenamiento, y_temporal = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

#dividimos el 30% temporal en partes iguales para validacion y prueba

X_validacion, X_prueba, y_validacion, y_prueba = train_test_split(
    X_temporal,
    y_temporal,
    test_size=0.50,
    random_state=42,
    stratify=y_temporal
)

In [12]:
#verificamos el tamaño de cada conjunto

resumen_conjuntos = pd.DataFrame({
    "conjunto": [
        "Entrenamiento",
        "Validacion",
        "Prueba"
    ],
    "documentos": [
        len(X_entrenamiento),
        len(X_validacion),
        len(X_prueba)
    ]
})

resumen_conjuntos["porcentaje_%"] = (
    resumen_conjuntos["documentos"]
    / len(corpus)
    * 100
).round(2)

display(
    resumen_conjuntos.style
    .hide(axis="index")
    .set_caption("Distribución de documentos por conjunto")
)

conjunto,documentos,porcentaje_%
Entrenamiento,795,69.920000
Validacion,171,15.040000
Prueba,171,15.040000


In [13]:
#comparamos la distribucion de categorias entre los tres conjuntos

distribucion_categorias = pd.concat(
    [
        y_entrenamiento.value_counts().rename("Entrenamiento"),
        y_validacion.value_counts().rename("Validacion"),
        y_prueba.value_counts().rename("Prueba")
    ],
    axis=1
).fillna(0).astype(int)

display(
    distribucion_categorias.style
    .set_caption("Cantidad de documentos por categoría")
)

,Entrenamiento,Validacion,Prueba
Type,,,
Macroeconomia,223,48,48
Alianzas,172,37,37
Innovacion,106,23,23
Regulaciones,99,21,21
Otra,90,19,20
Sostenibilidad,87,19,18
Reputacion,18,4,4


In [14]:
#comparamos los porcentajes de cada categoria para verificar la estratificacion

distribucion_porcentual = pd.concat(
    [
        y_entrenamiento.value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("Entrenamiento_%"),

        y_validacion.value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("Validacion_%"),

        y_prueba.value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("Prueba_%")
    ],
    axis=1
)

display(
    distribucion_porcentual.style
    .set_caption("Distribución porcentual de categorías")
)

,Entrenamiento_%,Validacion_%,Prueba_%
Type,,,
Macroeconomia,28.050000,28.070000,28.070000
Alianzas,21.640000,21.640000,21.640000
Innovacion,13.330000,13.450000,13.450000
Regulaciones,12.450000,12.280000,12.280000
Otra,11.320000,11.110000,11.700000
Sostenibilidad,10.940000,11.110000,10.530000
Reputacion,2.260000,2.340000,2.340000


El conjunto de entrenamiento contiene los documentos que el modelo utiliza para aprender los patrones que permiten relacionar el texto de una noticia con su categoria. El conjunto de validacion contiene documentos que no se utilizan directamente para entrenar el modelo. Este conjunto sirve para comparar distintas configuraciones y tomar decisiones durante el desarrollo, por ejemplo seleccionar parametros o determinar cual modelo produce mejores resultados. Por ultimo, el conjunto de prueba se utiliza para realizar la evaluacion final del modelo con documentos que no fueron utilizados durante el entrenamiento ni para tomar decisiones durante la validacion. No se debe utilizar este conjunto para ajustar el modelo porque entonces los resultados dejarian de representar correctamente su desempeño sobre datos realmente desconocidos.